In [ ]:
# 두개의 하위 팀으로 구성된 멀티 에이전트 네트워크를 통해 경제 데이터를 조사 및 수집하고 이를 시각화 한 뒤 보고서를 자동으로 작성하는 시스템을 구현함.
# 리서치 팀(Research Team)은 웹 검색과 웹 스크래핑을 담당해 필요한 데이터를 수집하며, 문서 작성 팀(Writing Team)은 보고서의 개요 작성, 본문 생성, 그리고 차트 생성을 수행함.
# 이 두 팀은 각각 독립적인 서브 그래프로 구현되고, 최상위 감독(Supervisor) 노드가 작업의 순서와 흐름을 조율함.
# 이번 프로젝트에서는 웹 페이지 내용을 읽기 위해 beautifulsoup4 라이브러리가 필요하므로 다음과 같이 사전에 설치함.
# pip install beautifulsoup4

# 01. 상태 정의
# 먼저 필요한 라이브러리를 업로드하고, 멀티 에이전트 네트워크 전반에서 공유할 상태를 정의함.
# 이 예제에서는 MessagesState 를 사용하지 않고, TypedDict 로 상태 구조를 직접 정의함. 
# 메시지 이력(messages)뿐 아니라 감독자가 선택한 다음 실행할 노드이름을 저장함. next 필드를 추가로 포함해야 하기 때문임.
# messages 필드는 add_messages 로 주석(Annotated) 처리되어 있어 각 노드가 반환하는 메시지가 기존 메시지 리스트에 누적되도록 동작함.
# 반면 next 필드는 실제 작업 데이터를 담기 위한 값이 아니라 조건부 분기(add_conditional_edges)에서 참조되는 라우팅 정보로 사용됨.
# 즉, 감독자 노드가 next 값을 설정하고, 그래프는 이 값을 기반으로 다음 실행 노드를 결정하며 흐름을 이어감.

from __future__ import annotations
from typing import Annotated, List, Dict, Optional, Literal
from typing_extensions import TypedDict

from pathlib import Path
from pydantic import BaseModel, Field

# Langgraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Command

# LangChain
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, BaseMessage, SystemMessage
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from langchain_community.document_loaders import WebBaseLoader

# 1) 공용 상태 정의
class State(TypedDict):
    messages : Annotated[List[BaseMessage], add_messages]
    next: str

####################################################
# from __future__ import annotations 
####################################################

# 파이썬의 타입 힌트(Type Hint) 평가 시점을 런타임이 아닌 정적 분석 시점으로 미뤄주는 기능임.
# 이 코드를 최상단에 선언하면 파이썬은 타입 힌트를 실제 객체가 아닌 단순한 문자열 로 취급함.
# 주요 사용 목적과 장점.
#   1. 전방 참조(Forward Reference) 문제 해결
#   파이썬은 위에서 아래로 코드를 읽기 때문에, 아직 정의되지 않은 클래스를 타입 힌트에 쓰면 오류(Name Error)가 발생함.
#   . 적용 전(오류 발생)
#   --------------------------------------------------------------------------
#       class User:
#           # 에러 User 클래스가 아직 완전히 정의되지 않았기 때문임
#           def clone(self) -> User:
#               pass
#   --------------------------------------------------------------------------
#   . 적용 후(정상 작동)
#   --------------------------------------------------------------------------
#       from __future__ import annotations
#       class User:
#           # 문자열로 취급되므로 에러 없이 정상 작동함.
#           def clone(self) -> User:
#               pass
#   --------------------------------------------------------------------------

#   2. 실행 속도 향상 및 메모리 절약
#   이 구문이 없으면 파이썬은 모듈을 임포트하는 시점에 타입 힌트에 적힌 모든 객체를 실제로 생성하고 평가함.
#   하지만 annotations 를 임포트하면 이를 모두 문자열로 보관하므로, 초기 로딩 속도가 빨라지고 런타임 메모리 오버헤드가 줄어듬

#   3. 최신 타입 문법 하위 호환성 지원
#   파이썬 3.10부터 List[int] 대신 내장 generic인 list[int]를 사용할 수 잇고, Union 대신 int | str 같은 파이프 연산자(|)를 지원함.
#   최신 파이썬 버전을 지원하는 환경에서 이 구문을 쓰면, 구버전 파이썬 환경에서도 문법 에러 없이 코드를 파싱할 수 있게 도와줌. 

####################################################
# from typing import Annotated
####################################################

# 파이썬 3.9에 도입된 기능으로, 기존 타입 힌트에 개발자가 원하는 임의의 메타데이터(부가 정보)를 추가할 수 있게 해주는 기능임.
# 타입 시스템 자체를 방해하지 않으면서 코드의 의미를 풍부하게 만들어 줌.

# 기본 문법 구조
# ---------------------------------------------------------------------------------
#   from typing import Annotated
#   # 변수명 Annotated[원래_타입, 메타데이터1, 메타데이터2, ...]
#   age: Annotated[int, "0세 부터 12세 사이의 정수"] = 25
# ---------------------------------------------------------------------------------
#   . 첫번째 인자: int, str 같은 실제 파이썬 타입이 옴.
#   . 두번째 이후 인자: 문자열, 객체, 함수 등 어떤 형태의 메타데이터든 자유롭게 넣을 수 있음.

# 주요 활용 사례
# 정적 타입 검사기(MyPy 등)는 메타데이터를 무시하고 첫번째 타입만 검사하지만, 런타임 프레임워크들은 이 메타데이터를 읽어서 강력한 기능을 수행함.
# 1. FastAPI/ Pydantic(데이터 검증 및 의존성 주입)
#   현대 파이썬 웹 개발에서 가장 많이 쓰이는 형태임. 데이터 검증 규칙이나 의존성을 타입에 심을 수 있음.
# ---------------------------------------------------------------------------------
#   from fastapi import Depends, Query
#   from typing import Annotated

#   # 1. 데이터 검증: 글자수가 3~10자 이어야 하는 아이디 타입 정의
#   UserId = Annotated[str, Query(min_length=3, max_length=10)]
# --------------------------------------------------------------------------------- 

####################################################
# from typing_extensions import TypedDict
####################################################

# 파이썬 딕셔너리(Dictionary)의 내부 Key 와 Value 에 대해 고정된 이름을 가진 타입을 지정할 수 있게 해주는 기능임.
# 일반적인 Dict[str, Any]는 어떤 Key가 들어있는지 알수 없지만, TypedDict를 사용하면 딕셔너리 구조를 클래스 처럼 명확하게 정의할 수 있음.

# 왜 typing 이 아니라 typing_extensions 인가?
#   TypedDict 는 파이선 3.8버전에 표준 라이브러리(typing)로 정식 도입됨.
#   만약 프로젝트가 3.8 미만 버전을 지원해야 하거나, 최신 버전에 추가된 TypedDict 를 구버전에서도 안전하게 사용하고 싶을 때 하위 호환성 패키지인 typing_extensions 를 가져와 사용
#   3.8 이상을 지원하는 환경이라면 from typing import TypedDict 로 작성해도 무방함.

# 1. 기본 사용법
# 딕셔너리가 가질 Key 이름과 Key 의 데이터 타입을 클래스 형태로 지정함.
# ----------------------------------------------------------------------------------------
#   from typing_extensions import TypedDict
#   # 1. 딕셔너리 구조 정의
#   class UserProfile(TypedDict):
#       name: str
#       age: int
#       is_active: bool
# 2. 선언된 타입에 맞게 딕셔너리 생성
#   user: UserProfile = {
#       "name": "홍길동",
#       "age": 30,
#       "is_active": True
#   }
# ----------------------------------------------------------------------------------------

# 2. 필수값과 선택값 지정.
#   a. 전체를 선택사항으로 만들기
# ----------------------------------------------------------------------------------------
#   class UserProfile(TypedDict, total=False):
#       name: str # 넣어도 되고 안넣어도 됨
#       age: int  # 넣어도 되고 안넣어도 됨  
# ----------------------------------------------------------------------------------------

#   b. 특정 Key 만 선택 사항으로 만들기(NotRequired) 3.11 버전 이상에서 지원
# ----------------------------------------------------------------------------------------
#    from typing_extensions import TypedDict, NotRequired
#    class UserProfile(TypedDict):
#       name: str
#       age: int
#       email: NotRequired[str] # 선택
# ----------------------------------------------------------------------------------------

# 3. 다른 도구들과의 차이점.
#   . TypedDict | 순수한 파이썬 DICT | 단순 데이터 구조 표현에 좋음. JSON 변환(Json.dumps)이 바로 가능하며 오버헤드가 없음. (런타임 검증 기능이 없음)
#   . NamedTuple | 튜플 tuple | 불변(Immutable) 데이터이며, user.name 처럼 점(.)으로 접근함.
#   . Dataclass | 커스텀 클래스 객체 | 객체 지향적인 메서드가 필요하거나, 기본값 설정이 복잡할 때 씀
#   . Pydantic(BaseModel) | 커스텀 클래스 객체 | FastAPI 등으로 쓰이며, 타입 힌트를 기반으로 런타임에 들어온 값을 실제로 검증(Validation)하고 변환해 줌.

# 정리하자면, API 요청/응답으로 주고받는 JSON 형태의 딕셔너리에 타입 안전성과 자동 완성을 제공하고 싶을때 가장 가볍고 강력하게 쓸수 있는 도구임.

####################################################
# from pathlib import Path
####################################################

# 파이썬에서 파일이나 디렉토리(폴더)의 경로를 다룰 때 사용하는 가장 표준적이고 현대적인 도구임.
# 과거에는 파일 경로를 문자열(str)로 다루며 os.path.join() 등을 썻지만, 파이썬 3.4 부터는 경로 자체를 직관적인 하나의 '객체'로 다루는 pathlib 사용이 권장됨.

# 왜 os.path 대신 Path 를 써야 할까요?
#   1. 운영체제(OS) 독립성 : 윈도우(\)와 맥/리눅스(/)는 경로 구분자가 다름. Path를 쓰면 코드가 실행되면 OS에 맞춰 구분자를 자동으로 처리해 줌.
#   2. 슬래시(/) 연산자 지원 : 직관적으로 경로를 합칠 수 있음.
#   3. 가독성 향상 : 문자열 함수를 복잡하고 중첩하지 않고, 점(. 변수접근)방식으로 깔끔하게 짤 수 있음.

# 핵심 사용법.
#   1. 경로 생성 및 결합
#   --------------------------------------------------------------------------------------------------------
#       from pathlib import Path
#       # 현재 작업 디렉토리 기준 경로 객체 생성
#       base_dir = Path(".")
#       # / 연산자 폴더와 파일 경로를 직관적으로 합치기
#       config_file = base_dir / "config" / "setting.json"
#       
#       print(config_file) # 실행중인 OS 에 맞춰 자동으로 문자열 출력(예: config/setting.json)            
#   --------------------------------------------------------------------------------------------------------

#   2. 파일/폴더 정보 쉽게 가져오기
#   --------------------------------------------------------------------------------------------------------
#       file_path = Path("/users/project/date.tar.gz")
#
#       print(file_path.name)   # 'data.tar.gz' (전체 파일명)
#       print(file_path.stem)   # 'data.tar' (마지막 확장자를 뺀 이름)
#       print(file_path.suffix) # '.gz' (마지막 확장자)
#       print(file_path.suffixes)   # ['tar', 'gz'] (모든 확장자 리스트)
#       print(file_path.parent) # '/users/project' (부모 디렉토리)
#   --------------------------------------------------------------------------------------------------------

#   3. 파일 검사 및 생성/삭제(런타임 제어)
#   --------------------------------------------------------------------------------------------------------
#       path = Path("output/report.txt")
#
#       # 존재 여부  및 타입 확인
#       print(path.exists()) # True/False (존재하는가?)
#       print(path.is_file()) # 파일인가?
#       print(path.is_dir()) # 디렉토리 인가?

#       # 폴더가 없으면 상위 폴더 까지 한번에 만들기
#       path.parent.mkdir(parents=True, exist_ok=True)

#       # 파일 생성 및 쓰기/읽기 (open 문 없이 한줄로 가능)
#       path.write_text("Hello world", encoding="utf-8")
#       content = path.read_text(encoding="urf-8")
#   --------------------------------------------------------------------------------------------------------

#   4. 파일 찾아내기(Globbing)
#   특정 패턴을 가진 파일들을 손쉽게 검색할 수 있음.
#   --------------------------------------------------------------------------------------------------------
#       current_dir = Path(".")
# 
#       # 현재 폴더에서 .py 로 끝나는 모든 파일 찾기
#       python_files = current_dir.glob("*.py")
#       
#       # 하위 폴더 전체를 뒤져서 .csv 파일 모두 찾기(재귀 검색)
#       all_csv_files = current_dir.rglob("*.csv")    
#   --------------------------------------------------------------------------------------------------------

#   기존 os.path 문법과의 직관적 비교
# 
#   경로 합치기     |       os.path.join(a, b)      |       a / b
#   파일명만 추출   |       os.path.basename(path)  |       path.name
#   부모 폴더 찾기  |       os.path.dirname(path)   |       path.parent
#   절대 경로 변환  |       os.path.abspath(path)   |       path.resolve()

####################################################
# from langgraph.graph import StateGraph
####################################################

# LangChain 생태계의 대표적인 프레임워크인 LangGraph에서 에이전트(Agent)나 워크플로우를 그래프 구조로 설계할 때 사용하는 핵심 클래스임.
# 대규모 언어 모델(LLM)을 기반으로 복잡한 반복(Loop), 조건부 분기(Conditional Branching), 상태 유지(State Management)가 필요한 애플리케이션을 만들때 구조를 짜는 '도화지'역할을 함.

# 기존의 LangChain 이 일직선으로 진행되는 체인 구조였다면 LangGraph 는 노드와 간선(Edge)으로 이루어진 순환 그래프(Cyclic Graph)를 만듬
# 이때 가장 중요한 것이 상태(State)임. StateGraph 는 전체 워크플로우가 진행되는 동안 중앙에서 하나의 데이터를 상태(State)를 유지하고 공유함.
# 각 단계(Node)는 이 상태를 읽어서 행동하고, 완료 후 상태를 업데이트함.

####################################################
# from pydantic import BaseModel, Field
####################################################

# 파이썬에서 데이터의 형태를 정의하고, 들어오는 값이 올바른지 자동으로 검증(Validation)하기 위해 사용하는 Pydantic 라이브러리의 핵심 도구임.
# FastAPI 를 비롯한 현대 파이썬 웹 프레임워크나 LangChain 같은 AI 라이브러리에서 데이터 가이드라인을 잡을 때 필수적으로 사용함.

# BaseModel 과 Field 의 역할 분담
#   . BaseModel : 검증하고 싶은 데이터 구조의 큰 틀(클래스)를 정의함.
#   . Field : 그 틀 안에 들어가는 개별 필드(속성)의 세부 규칙과 메타데이터를 지정함.

# 핵심 사용법
# 두 도구를 어떻게 조합하여 사용하는지 직관적인 예시 코드로 살펴봄.

#   --------------------------------------------------------------------------------------------------------
#       from pydantic import BaseModel, Field
#       from typing import Optional

#       1. BaseModel 을 상속받아 데이터 구조(스키마) 정의
#       class UserRegisterForm(BaseModel):
#           # Field를 사용하지 않은 기본 선언(타입만 검사)
#           username: str

#           # Field 를 사용해 세수 제약 조건 추가
#           # 최소 3자, 최대 20자 제한 및 Swagger 문서용 설명 추가
#           nickname: str = Field(
#               min_length=3,
#               max_length=20,
#               description="사용자에게 표시될 별명"            
#           )

#           # 숫자 번위 제한 및 기본값(Default) 설정
#           age: int = Field(
#               default = 20,
#               ge = 0,
#               le = 120,
#               description = "나이는 0세 부터 120세 까지만 허용"
#           )

#       2. 올바른 데이터 입력 시(정상 작동)
#       valid_data = {
#           "username": "charlie",
#           "nickname": "코딩하는 고양이",
#           "age": 20,
#           "ID": "user_12345"
#       }
#       user = UserRegisterForm(**valid_data)
#       print(user.nickname) # 출력 : 코딩하는 고양이
#       print(user.user_id) # 출력 : user_12345 (alias 덕분에 'ID'키를 찾아서 매핑함)

#       3. 잘못된 데이터 입력 시(런타임 에러 발생)
#       try :
#           invalid_data = {
#               "username" : "charlie",
#               "nickname" : "나", # 에러 : min_length(3자) 미달
#               "age" : 150,       # 에러 : le(120세) 초과
#               "ID" : "user12345"
#           }
#           UserRegisterForm(**invalid_data)
#       except ValidationError as e:
#           print(e.json()) # 데이터의 어느 부분이 왜 틀렸는지 정확하게 알려줌.   
#   --------------------------------------------------------------------------------------------------------

# Field의 주요 기능과 인자(Arguments)들
# Field 함수 안에는 데이터 검증과 문서화를 위한 강력한 옵션들이 준비되어 있음.
# 1. 데이터 검증(Constraints)
#   . 문자열 제약: min_length(최소길이), max_length(최대 길이), pattern(정규표현식 매칭)
#   . 숫자 제약: gt(GreaterThan), ge(GreaterEqual, 이상), lt(LessThen, 미만), le(LessEqual, 이하)
#   . 리스트 제약: min_items(최소 원소 개수), max_items(최대 원소 개수)

# 2. 메타데이터 및 문서화
#   . discription : 해당 필드가 무엇을 뜻하는지 적어두면, FastAPI 가 이를 읽어 Swagger API 문서에 자동으로 설명을 더 해줌.
#   . examples: 예시 데이터를 리시트 형태로 넣어 문서 가독성을 높임.

# 3. 필드 제어
#   . default : 값이 전달되지 않았을 때 사용할 기본값임
#   . default_factory : datetime.now나 list 처럼 매번 새로운 객체를 동적으로 생성해야 하는 기본값이 필요할 때 함수를 지정함. (default_factory = list)
#   . alias : JSON 데이터의 키 이름(예: snake_case가 아닌 camelCase)을 파이썬 변수명과 매핑해 줌.

# from typing import Annotated 와의 결합
# Pydantic v2 부터는 Field 를 Annotated 와 결합하여 재사용 가능한 타입 묶음을 만들수 있음.
# ------------------------------------------------------------------------------------------------
#   from typing import Annotated
#   
#   # 0~120세 제한 규칙을 하나의 독립된 타입으로 정의
#   AgeType = Annotated[int, Field(ge=0, le=120)]
#
#   class Profile(BaseModel):
#       user_age: AgeType # 정의해둔 규칙이 그대로 적용됨
# ------------------------------------------------------------------------------------------------

#################################################################
# FastAPI, Swagger API
#################################################################

# FastAPI 는 파이썬 표준 타입 힌트를 기반으로 백엔드 API를 빠르고 손쉽게 만들 수 있게 해주는 고성능 현대식 웹 프레임워크임.
# Swagger UI는 이 FastAPI 가 만든 API 구조를 시각적인 웹 페이지로 보여주고 브라우저에서 즉시 테스트할 수 있게 돕는 대화형 API 문서화 도구임.

# 두 도구는 FastAPI 안에서 완전히 하나로 융합되어 작동하므로, 개발자가 API 코드를 짜면 문서가 실시간으로 자동 생성됨.

# 1. FastAPI : 무엇이 특별한가?
#   . 압도적인 속도(High Performance): 내장된 ASGI 서버(Starlette) 덕분에 내부적으로 async/await 비동기 처리를 완벽히 지원하며, 
#                                       Node js 나 Go 언어에 비견될 만큼 파이썬 프레임워크 중 가장 빠름
#   . 자동 데이터 검증: 앞서 살펴본 Pydantic 과 결합하여, 클라이언트가 데이터를 잘못 보내면 알아서 422 Unprocessable Entity 에러와 함께 원인을 짚어줌.
#   . 직관성과 생산성 : 파이썬의 표준 타입 힌트(name: str, age: int)만 잘 적어주면 코드 자동 완성, 타입 체킹이 완변하게 지원되어 개발 속도가 2~3배 빨라짐.

# 2. Swagger UI : API 문서화의 혁신
#   과거엔 백엔드 코드를 수정하면 워드나 Notion, Postman 에 API 스펙을 수동으로 업데이트해야해서 코드가 바뀌면 문서가 밀리는 현상이 잦았음
#   . 코드 기반 자동 동기화 : FastAPI 가 코드를 해석해 내부적으로 OpenAPI 표준 명세서(JSON)를 만들고, 이를 바탕으로 Swagger 웹 화면을 무료로 자동 구축함.
#   . 즉각적인 대화형 테스트(Try it out) : 별도의 API 테스트 프로그램(Postman 등)을 켤 필요없이, 
#                                       웹 브라우저 화면에서 직접 파라미터나 JSON 바디를 입력해 Execute 버튼을 누르면 실제 서버 응답을 눈으로 확인할 수 있음.

# 3. 한눈에 보는 연동 예제.
# FastAPI 에서 두 도구가 얼마나 자연스럽게 연동되는지 보여주는 예시임.
# -----------------------------------------------------------------------------------------------------------------------------
#   from fastapi import FastAPI, Field
#   from pydantic import BaseModel
#   app = FastAPI(
#       title = "상품 관리 API 시스템",
#       description = "FastAPI와 Swagger가 제공하는 자동화 문서 예제입니다.",
#       version = "1.0.0" 
#   )

#   1. Pydantic 으로 요청 데이터 구조 정의
#   class Product(BaseModel):
#       name: str = Field(..., description="상품의 이름", example="무선 마우스")
#       price: int = Field(..., ge=0, description="상품 가격 (0원 이상)", example=35000)

#   2. API 엔드포인트 작성
#   @app.post("products/", status_code=201, tags=["상품 관리 API"])
#   def create_product(product: Product):
#       """
#       새로운 상품을 시스템에 등록합니다.
#       - **name**: 필수 입력 항목
#       - **price**: 0원 이상 입력 필수
#       """
#       return  {"message": "등록 성공", "data": product}
# -----------------------------------------------------------------------------------------------------------------------------

# 결과 확인 방법
#   위 코드를 실행(fastapi dev main.py 또는 uvicorn)한 뒤 브라우저를 연다.
#   1. http://127.0.0.1:8000/docs 에 접속
#   2. 화면에 [상품 관리 API]라는 태그로 그룹화된 깔끔한 UI 가 나타남
#   3. Product 모델에 적어둔 description ("상품의 이름")과 example 데이터가 Swagger 화면 속  설명 란에 그대로 채워지는 것을 볼수 있음.
#   4. Try it out 버튼을 눌러 가격에 -100을 넣고 테스트 하면, Swagger 화면에 Pydantic 이 걸러낸 에러 메시지가 예쁘게 표시됨.

# 보너스: ReDoc 지원
# FastAPI 는 Swagger UI 외에도 또 다른 깔끔한 레이아웃의 문서 도구인 ReDoc 를 기본 내장하고 있음.
# 서버를 켠 사앹로 http://127.0.0.1:8000/redoc 에 접근하면 대규모 API 명세서를 한눈에 읽기 좋은 3단 구조의 정적인 문서 페이지를 확인가능함.

####################################################
# from langgraph.types import Command
####################################################

# LangGraph 에서 노드(Node)내부의 실행 로직안에서 그래프의 상태(State) 업데이트와 다음 이동할 노드(Routing)를 동시에 제어할 수 있게 해주는 도구임.
# 기존 LangGraph 는 그래프 구조를 정의할 때 조건부 간선(add_conditional_edges)을 미리 밖에서 선언해 두어야 했음.
# 하지만 Command 를 사용하면 "간선(Edge)없는 유연한 그래프(Edgeless Graph)"를 구현할수 있어, 노드가 실행중에 상황에 따라 다음 목적지를 동적으로 직접 바꿀 수 있음.

# 핵심 인자(Arguments)
#   1. update : 현재 그래프 상태(State) 에 반영할 데이터를 딕셔너리 형태로 전달함.
#   2. goto : 다음에 실행할 노드의 이름을 문자열로 지정함.
#   3. resume : interrupt() 함수와 결합하여 사용자의 입력을 받아 그래프를 다시 재개할 때 사용함.

# 주요 활용 예시
#   1. 조건부 간선(Conditional Edge)없이 동적 라우팅 구현히기
#       노드 내부 로직 결과에 따라 분기 처리를 즉시 처리하는 가장 대표적인 패턴임.
#       상태 값에 따라 Command(update=..., goto=...) 를 반환하여 add_conditional_edges 없이도 if-elif 문 만으로 다음 노드를 유연하게 지정할 수 있음.
#       장점 : 복잡한 조건부 간선 설계를 생략하고 노드 함수 내부에서 직관적으로 흐름을 제어함.
#   2. Human-in-the-loop(인간 개입 및 재개) 구현하기
#       AI 작업 중 interrupt()로 멈춘 뒤 사용자의 입력을 받아 Command(resume=...)로 그래프를 재개할수 있음. 

#####################################################################################
# from langchain_core.messages import HumanMessage, BaseMessage, SystemMessage
#####################################################################################

# LangChain 및 LangGraph 생태계에서 대규모 언어 모델(LLM)과 주고받는 대화 데이터를 표준화된 객체 형태로 다루기 위한 핵심 클래스들임.

# 1. 계층 구조와 핵심 개념.
#   이 클래스 들은 부모와 자식 관계를 가짐.
#   . BaseMessage : 모든 메시지 클래스의 최상위 부모 클래스임. 직접 생성해서 쓰기보다는 함수에서 여러 종류의 메시지를 한번에 타입 힌트로 지정할 때 주로 사용 (messages: list[BaseMessage])
#   . HumanMessage : 사용자(Human)가 LLM에게 보내는 요청이나 질문 메시지임.
#   . SystemMessage : LLM에게 페르소나(역할), 규칙, 지침 을 부여하는 시스템 프롬프트 메시지임.

# 2. 주요 속성.
#   모든 메시지 객체는 BaseMessage 로부터 상속 받은 공통 속성을 가지고 있음.
#   . content : 메시지의 실제 텍스트 내용임. (문자열 도는 멀티 모달용 리스트)
#   . id : 메시지를 식별하기 위한 고유 ID임. 
#   . name : 메시지를 보낸 주체의 이름임 
#   . additional_kwargs: 특정 LLM 모델 고유의 추가 데이터(예: OpenAI의 Tool Call 정보 등)를 담는 딕셔너리임.


In [ ]:
# 02. 리서치 도구 정의
# 리서치 팀은 두가지 일이 핵심임
# (1) 검색 엔진으로 관련 링크를 찾고, (2) 그 링크들을 실제로 열어 본문 텍스트를 추출함. TavilySearch 가 1번을 WebBaseLoader가 2번을 담당함.

# Tavily 웹 검색 : 상위 n개 결과를 반환.
# Tavily 는 웹 페이지 제목, URL, 요약문 등의 메타데이터를 함께 제공하므로 이후 에이전트가 필요한 문서만 선별해서 사용할 수 있음.
tavily_tool = TavilySearch(max_result=5)

# scrape_webpage 는 특정 URL 목록을 입력받아 각 웹페이지의 본문 텍스트를 추출함.
# 여기서는 WebBaseLoader 를 사용하여 요청 시 User-Agent 헤더를 명시적으로 설정해 크롤링 시 브라우저 환경과 유사하게 보이게 함.
# 이는 일부 웹서버가 기본 파이썬 요청을 차단하는 경우를 방지하기 위함.
@tool 
def scrape_webpage(urls: List[str]):
    """요청한 웹페이지 목록을 크롤링하여 본문 텍스트를 추출합니다."""
    loader = WebBaseLoader(
        urls,
        header_template = [
            "user-agent": (
                "Mozilla/5.0 (Window NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            )
        ]
    )

    docs = loader.load()

    return "\n\n".join(
        [
            f'<문서 제목 = "{doc.metadata.get("title", "")}">\n{doc.page_content}\n</문서>'
            for doc in docs
        ]
    )